## Akış (blok başına): x → Self-Attention → Add&Norm → FFN → Add&Norm → çıktı (aynı şekil).

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0, "d_model, h'ye tam bölünmeli"
        self.h = h
        self.d_k = d_model // h
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # tüm kafaların W_Q'su yan yana
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # çıkış projeksiyonu

    def forward(self, x_q, x_kv, mask=None):
        # x_q: (B, n_q, d_model)   x_kv: (B, n_k, d_model)
        B, n_q, _ = x_q.shape
        n_k = x_kv.shape[1]
        # 1) Projeksiyon sonra kafalara böl
        Q = self.W_Q(x_q).view(B, n_q, self.h, self.d_k).transpose(1, 2)   # (B, h, n_q, d_k)
        K = self.W_K(x_kv).view(B, n_k, self.h, self.d_k).transpose(1, 2)  # (B, h, n_k, d_k)
        V = self.W_V(x_kv).view(B, n_k, self.h, self.d_k).transpose(1, 2)  # (B, h, n_k, d_k)
        # 2) Her kafada ölçekli dot-product attention
        scores = Q @ K.transpose(-2, -1) / self.d_k ** 0.5                 # (B, h, n_q, n_k)
        if mask is not None:
            scores = scores + mask                                         # (n_q, n_k) yayınlanır
        weights = F.softmax(scores, dim=-1)                                # (B, h, n_q, n_k)
        heads = weights @ V                                                # (B, h, n_q, d_k)
        # 3) Concat: kafaları yan yana birleştir
        concat = heads.transpose(1, 2).contiguous().view(B, n_q, -1)       # (B, n_q, h*d_k = d_model)
        # 4) W_O
        return self.W_O(concat), weights                                   # (B, n_q, d_model)

torch.manual_seed(0)
mha = MultiHeadAttention(d_model=512, h=8)
x = torch.randn(2, 10, 512)                  # batch=2, n=10 token
out, w = mha(x, x)
print("çıktı:", out.shape)
print("ağırlıklar:", w.shape)
print("parametre sayısı:", sum(p.numel() for p in mha.parameters()))

çıktı: torch.Size([2, 10, 512])
ağırlıklar: torch.Size([2, 8, 10, 10])
parametre sayısı: 1048576


In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.lin1 = nn.Linear(d_model, d_ff)      # W1, b1
        self.lin2 = nn.Linear(d_ff, d_model)      # W2, b2
    def forward(self, x):                         # x: (B, n, d_model)
        return self.lin2(torch.relu(self.lin1(x)))  # (B,n,d_ff) -> (B,n,d_model)

# Elle sayısal örneğimiz
ffn_small = FeedForward(2, 3)
with torch.no_grad():
    ffn_small.lin1.weight.copy_(torch.tensor([[1., 0.], [0., 1.], [2., -1.]]))  # (3,2) = W1^T
    ffn_small.lin1.bias.zero_()
    ffn_small.lin2.weight.copy_(torch.tensor([[1., 0., 1.], [0., 1., 1.]]))     # (2,3) = W2^T
    ffn_small.lin2.bias.zero_()
print(ffn_small(torch.tensor([[1., -1.]])))       # beklenen [4, 3]

# Gerçek boyut
ffn = FeedForward(512, 2048)
x = torch.randn(2, 10, 512)
print(ffn(x).shape)

# Pozisyon bağımsızlığı: bir token'ın çıktısı diğerlerine bağlı değil
y_all = ffn(x)
y_one = ffn(x[:, 3:4, :])
print(torch.allclose(y_all[:, 3:4, :], y_one, atol=1e-6))

tensor([[4., 3.]], grad_fn=<AddmmBackward0>)
torch.Size([2, 10, 512])
True


In [5]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model=512, h=8, d_ff=2048):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, h)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # Adım 1: attention (iletişim) + Add&Norm
        attn_out, w = self.attn(x, x, mask)       # (B,n,d_model)
        x = self.norm1(x + attn_out)              # (B,n,d_model)
        # Adım 2: FFN (işlem) + Add&Norm
        x = self.norm2(x + self.ffn(x))           # (B,n,d_model)
        return x, w

class TransformerStack(nn.Module):
    def __init__(self, N=6, d_model=512, h=8, d_ff=2048):
        super().__init__()
        self.blocks = nn.ModuleList([TransformerBlock(d_model, h, d_ff) for _ in range(N)])
    def forward(self, x, mask=None):
        for blk in self.blocks:                   # aynı yapı N kez
            x, _ = blk(x, mask)
        return x

torch.manual_seed(0)
stack = TransformerStack(N=6)
x = torch.randn(2, 10, 512)
y = stack(x)
print("girdi:", x.shape, "-> çıktı:", y.shape)
print("toplam parametre:", f"{sum(p.numel() for p in stack.parameters()):,}")

girdi: torch.Size([2, 10, 512]) -> çıktı: torch.Size([2, 10, 512])
toplam parametre: 18,902,016
